# Wake word training — **"Hey, Cee Jap"**

Trains an [openWakeWord](https://github.com/dscripka/openWakeWord) model for the
CJ Panganiban museum kiosk and exports `hey_cee_jap.onnx`, ready to drop into
`config.WAKE_OWW_MODEL_PATH`.

## Why openWakeWord and not a speech model

| Option | Verdict |
|---|---|
| **openWakeWord** ✅ | Trains a ~100 KB classifier head on top of Google's frozen `speech_embedding` extractor. Minutes-to-hours on one GPU, runs always-on at ~1% CPU on a Pi, exports ONNX. `auto_train` optimises directly for **false-accepts-per-hour** — the number that goes in front of stakeholders. |
| Whisper / Wav2Vec2 fine-tune | Wrong tool. Hundreds of MB, ~1 s latency, needs a GPU to stay resident. Fine for transcription, unusable as an always-on trigger. |
| Porcupine | Better accuracy, but custom phrases need a paid commercial licence. |
| microWakeWord | Aimed at ESP32 microcontrollers, not a Pi-class board. |

It is also what the app already expects: [`app/wake_word.py`](../../../app/wake_word.py)
has an `OpenWakeWordDetector` backend wired to `config.WAKE_OWW_MODEL_PATH`, and
both `validate.py` and `test_detection.py` load `.onnx` via
`inference_framework="onnx"`.

## How the 34 real recordings are used

**As a test set, never as training data.** openWakeWord learns from tens of
thousands of synthetic Piper-TTS utterances with room-impulse and background-noise
augmentation. 34 clips from 3 speakers would be swamped in training, but they are
the only *real* evidence the model works — so they are held out and scored at the
end, broken down by SNR tier and speaker.

## Runbook

1. **Runtime → Change runtime type → GPU** (T4 is fine), then run cells top to bottom.
2. Leave `QUICK_SMOKE = True` for the first pass. It finishes in ~15 min and proves
   the whole chain. Only then set it to `False` and do the real run.
3. Colab disconnects after ~90 min idle. Keep the tab open, or mount Drive in step 3
   so a dropped session doesn't cost you the downloads.

---
## Step 0 — Confirm a GPU is attached

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "no nvidia-smi")

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> Hardware accelerator: GPU.\n"
        "CPU-only works but synthetic sample generation goes from minutes to many hours."
    )
print(f"torch {torch.__version__}   GPU: {torch.cuda.get_device_name(0)}")

---
## Step 1 — Install

**openwakeword is pinned to 0.6.0 and installed with `--no-deps`.** Its
dependency list demands `tflite-runtime` on Linux, which has no wheels for
Python ≥ 3.12 — so a plain `pip install openwakeword` makes the resolver
quietly *backtrack to 0.4.0*, a version with no `train.py` at all, and the
failure only surfaces as a `FileNotFoundError` at the patch cell before Step 8.
`--no-deps` skips that pin; the deps it actually needs are either preinstalled
on Colab (tqdm, scipy, scikit-learn, requests) or installed here
(`onnxruntime-gpu`, which also lets feature embedding run on the GPU). The
missing tflite runtime does not matter: the patch cell before Step 8 points
every feature-computation call at the equivalent `.onnx` models instead.

**`deep-phonemizer` is installed explicitly** because it is an *undeclared*
dependency: adversarial-negative generation does `from dp.phonemizer import
Phonemizer` for out-of-vocabulary words, but no openwakeword extra lists it —
without it, `--generate_clips` crashes halfway through, after the positives.

**`onnxscript` is required by torch ≥ 2.9's ONNX exporter** (it no longer
ships inside torch). Without it, training completes and then dies at the very
last step, exporting the model.

The trainer's wider tree (speechbrain, audiomentations, torchaudio, ...) can
still happily downgrade Colab's CUDA-enabled torch, so the cell after this one
re-checks that the GPU install is intact. If it fails,
**Runtime → Restart session** and re-run from Step 0 — a restart is usually all it needs.

In [ ]:
%pip install -q --no-deps openwakeword==0.6.0
%pip install -q onnxruntime-gpu deep-phonemizer onnxscript
%pip install -q torchinfo torchmetrics speechbrain audiomentations torch-audiomentations acoustics pronouncing mutagen datasets soundfile huggingface_hub pyyaml
print("installed")

In [ ]:
# Verify the install did not break CUDA torch.
# This must run in a fresh subprocess: the kernel imported torch in Step 0, and
# neither re-import nor importlib.reload() replaces the already-loaded C
# extensions - an in-process check reports the PRE-install state, so it can
# pass even after pip swapped in a CPU-only wheel (and Step 8's training
# subprocesses would then quietly run on CPU).
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True, text=True,
)
out = r.stdout.split()
assert r.returncode == 0 and out and out[-1] == "True", (
    "torch on disk lost CUDA -> Runtime > Restart session, then re-run from Step 0\n" + r.stderr
)

import torch
if torch.__version__ != out[0]:
    print(f"note: this kernel still holds torch {torch.__version__}; disk now has {out[0]}.\n"
          "      CUDA is intact, but restart the session if later imports misbehave.")

import pathlib
import openwakeword, speechbrain, audiomentations, torchinfo, torchmetrics, pronouncing, acoustics
assert (pathlib.Path(openwakeword.__file__).parent / "train.py").exists(), (
    "openwakeword has no train.py: pip resolved an old version instead of 0.6.0 "
    "(its tflite-runtime dep has no wheel for this Python). Re-run Step 1 exactly "
    "as written (--no-deps pin), then Runtime > Restart session and re-run from Step 0."
)
print(f"torch {out[0]} (CUDA ok)   openwakeword {getattr(openwakeword, '__version__', 'n/a')}")

---
## Step 2 — Piper sample generator

This is the TTS engine that produces the synthetic training utterances. It is a
separate repo from openWakeWord and is loaded by path, not by import.

**Pinned to the `v2.0.0` tag.** openwakeword 0.6.0 does
`from generate_samples import generate_samples` against the repo *root*
(`train.py:638`); upstream's `master` moved to a packaged v3 layout that has
neither that file nor `requirements.txt`. The cell below replaces a wrong
checkout if it finds one, and installs the generator's two missing dependencies
directly instead of using the repo's `requirements.txt` — whose `numpy<2` and
`audiomentations==0.33.0` pins would downgrade the Step 1 stack.

In [ ]:
import importlib.util, os, pathlib, shutil, subprocess, sys
os.chdir("/content")

REPO = pathlib.Path("/content/piper-sample-generator")

# openwakeword 0.6.0 imports generate_samples.py from the repo ROOT; only the
# v2.x tags have that layout. Replace a master/v3 checkout if one is present.
if REPO.exists() and not (REPO / "generate_samples.py").exists():
    print("master/v3 checkout detected - replacing with the v2.0.0 tag")
    shutil.rmtree(REPO)
if not REPO.exists():
    !git clone -q --depth 1 --branch v2.0.0 https://github.com/rhasspy/piper-sample-generator

# torch >= 2.6 defaults torch.load to weights_only=True, which refuses the
# pickled SynthesizerTrn object inside the v2.0.0 voice checkpoint. The file is
# the pinned release artifact from the rhasspy repo (size-checked below), so
# loading it fully is the intended behaviour.
gs = REPO / "generate_samples.py"
src = gs.read_text()
if "weights_only" in src:
    print("already patched: generate_samples.py torch.load")
elif "torch.load(model_path)" in src:
    gs.write_text(src.replace("torch.load(model_path)",
                              "torch.load(model_path, weights_only=False)"))
    print("patched generate_samples.py: torch.load(model_path, weights_only=False)")
else:
    print("WARNING: torch.load(model_path) not found in generate_samples.py "
          "(upstream changed? generation may fail on torch >= 2.6)")

VOICE = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"
VOICE_URL = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
if not pathlib.Path(VOICE).exists():
    !mkdir -p /content/piper-sample-generator/models
    !wget -q --show-progress -O {VOICE} {VOICE_URL}

# generate_samples.py needs exactly these two beyond Step 1 + stock Colab.
# Deliberately NOT `pip install -r requirements.txt`: its numpy<2 and
# audiomentations==0.33.0 pins would downgrade the Step 1 stack.
%pip install -q webrtcvad
if importlib.util.find_spec("piper_phonemize") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "piper-phonemize==1.1.0"])
    if r.returncode != 0:
        # No wheel for this Python version; community rebuild, same import name.
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "piper-phonemize-cross"], check=True)

size_mb = pathlib.Path(VOICE).stat().st_size / 1e6
assert size_mb > 50, f"voice checkpoint looks truncated ({size_mb:.1f} MB) - re-run this cell"

# Prove the whole import chain now rather than mid-Step-8.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from generate_samples import generate_samples
print(f"piper v2.0.0 ready: voice {size_mb:.0f} MB, generate_samples imports cleanly")

---
## Step 3 — Upload the real recordings

Upload **`cjap_clips.zip`** (built for you at `wakeword/CJAP/colab/cjap_clips.zip`).
It carries the 34 labelled positives, the 8 spelled-letter negatives, and
`validation_manifest.json`.

These are **held out** — they are scored in Step 9 and never trained on.

In [ ]:
import pathlib, zipfile, json, os
os.chdir("/content")

if not pathlib.Path("/content/data/validation_manifest.json").exists():
    from google.colab import files
    up = files.upload()                       # pick cjap_clips.zip
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall("/content")

manifest = json.load(open("/content/data/validation_manifest.json"))
c = manifest["counts"]
print(f"phrase in manifest : {manifest['phrase']}")
print(f"positives          : {c['positives']}   (easy {c['easy']} / moderate {c['moderate']} / hard {c['hard']})")
print(f"negatives          : {c['negatives']}")
print(f"speakers           : {', '.join(manifest['by_speaker'])}")

missing = [x["path"] for x in manifest["clips"] if not pathlib.Path("/content", x["path"]).exists()]
assert not missing, f"{len(missing)} clip(s) referenced by the manifest are missing, e.g. {missing[:3]}"
print(f"\nall {len(manifest['clips'])} clips present")

---
## Step 4 — Training parameters

**Everything you would want to tune is in this one cell.**

### On the negative phrases

`wakeword/README.md` warns *"confusable phrases belong in the EVALUATION set only,
never as training negatives"*. That warning is about adding a **handful of real
near-miss recordings** to a small training set, where they skew the decision
boundary. It does not apply here: openWakeWord's trainer is *designed* around
adversarial negatives (`generate_adversarial_texts(include_partial_phrase=1.0)`)
and generates thousands of them, balanced by `max_negative_weight`.

It also has to be this way. The **WW-5 decision (2026-07-27)** retired the spelled
`CJ` / `see jay` family and requires it to **stay silent**
([`app/wake_word.py:58`](../../../app/wake_word.py#L58)). A phrase that must not
fire has to be trained against. Your 8 real negatives are exactly that family —
they are the held-out check that this worked.

In [ ]:
MODEL_NAME = "hey_cee_jap"

# Spellings fed to Piper TTS. These are orthographic variants chosen so espeak-ng
# phonemises them to /heI si: dZaep/ - they are NOT mishear variants (those live in
# config.WAKE_PHRASE_VARIANTS on the STT backend, which is a different mechanism).
TARGET_PHRASES = [
    "hey cee jap",
    "hey see jap",
    "hey C jap",
    # Bare forms: visitors drop the carrier. Comment these out for a stricter,
    # lower-false-accept model that requires "hey" every time.
    "cee jap",
    "see jap",
]

CUSTOM_NEGATIVE_PHRASES = [
    # WW-5: the retired spelled-letter family MUST stay silent.
    "see jay", "cee jay", "C J", "CJ",
    "see jay ay pee", "cee jay ay pee", "C J A P",
    # Gallery vocabulary - spoken near the robot all day long.
    "chief justice", "chief justice panganiban", "panganiban",
    "CJ Panganiban", "the chief justice", "justice",
    # Acoustic neighbours of /si: dZaep/.
    "see japan", "japan", "cheap", "cheap app", "sea chap", "see chap",
    "see ya", "seagull", "jazz app", "the jab", "she chatted",
]

# Stakeholder metric. auto_train keeps the checkpoint that maximises recall while
# staying under this rate on the held-out false-positive validation audio.
TARGET_FP_PER_HOUR = 0.2

# ---- run size -------------------------------------------------------------
# First run: leave True. ~15 min, proves the chain end to end. The resulting model
# will be weak - that is expected and not a reason to change anything else.
QUICK_SMOKE = True

if QUICK_SMOKE:
    N_SAMPLES, N_SAMPLES_VAL, STEPS, AUG_ROUNDS = 1_000, 200, 2_000, 1
    N_BACKGROUND_CLIPS = 500
else:
    N_SAMPLES, N_SAMPLES_VAL, STEPS, AUG_ROUNDS = 30_000, 3_000, 50_000, 1
    N_BACKGROUND_CLIPS = 4_000

LAYER_SIZE = 32          # try 64 if recall on the 'hard' tier stays low
MODEL_TYPE = "dnn"

print(f"{'SMOKE TEST' if QUICK_SMOKE else 'FULL RUN'}: "
      f"{N_SAMPLES} positives, {STEPS} steps, target <= {TARGET_FP_PER_HOUR} FA/hr")
print(f"{len(TARGET_PHRASES)} target spellings, {len(CUSTOM_NEGATIVE_PHRASES)} custom negatives")

---
## Step 5 — Download the training corpora

Three things get pulled:

| What | Why | Size |
|---|---|---|
| `ACAV100M` precomputed features | The negative class — 2000 hours of general audio, already turned into embeddings | **~17 GB** |
| `validation_set_features.npy` | Held-out audio that `auto_train` measures false-accepts-per-hour against | ~2 GB |
| MIT RIRs + background clips | Room-impulse and noise augmentation, so the model survives a real gallery | ~1 GB |

> **These URLs have churned before** — the repo's own README documents AudioSet
> moving from tar to parquet mid-project. Every remote reference is in this one
> cell so there is a single place to fix. Step 6 verifies what actually landed.

The 17 GB file is the long pole. It is memory-mapped during training, not loaded,
so Colab's RAM is not the constraint — disk is. Check **Runtime → Manage sessions**
if you hit a disk error.

In [ ]:
import os, pathlib
os.chdir("/content")
pathlib.Path("/content/oww_data").mkdir(exist_ok=True)

from huggingface_hub import hf_hub_download

FEATURES_REPO = "davidscripka/openwakeword_features"
NEGATIVE_FEATURES_FILE = "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
VALIDATION_FEATURES_FILE = "validation_set_features.npy"

def fetch(repo, fname):
    print(f"fetching {fname} ...")
    p = hf_hub_download(repo_id=repo, filename=fname, repo_type="dataset",
                        local_dir="/content/oww_data")
    print(f"  -> {p}  ({pathlib.Path(p).stat().st_size/1e9:.2f} GB)")
    return p

NEGATIVE_FEATURES = fetch(FEATURES_REPO, NEGATIVE_FEATURES_FILE)
VALIDATION_FEATURES = fetch(FEATURES_REPO, VALIDATION_FEATURES_FILE)

In [ ]:
# Room impulse responses + background noise for augmentation.
import os, pathlib, numpy as np, soundfile as sf, scipy.signal
from datasets import load_dataset

RIR_DIR = pathlib.Path("/content/oww_data/mit_rirs")
BG_DIR  = pathlib.Path("/content/oww_data/backgrounds")
RIR_DIR.mkdir(parents=True, exist_ok=True)
BG_DIR.mkdir(parents=True, exist_ok=True)

def dump_16k(ds, outdir, limit, prefix, min_samples=16000):
    """Write a HF audio dataset out as 16 kHz mono wavs, capped at `limit`."""
    n = 0
    for row in ds:
        if n >= limit:
            break
        a = row["audio"]
        x, sr = np.asarray(a["array"], dtype=np.float32), a["sampling_rate"]
        if x.ndim > 1:
            x = x.mean(axis=1)
        if sr != 16000:
            x = scipy.signal.resample_poly(x, 16000, sr).astype(np.float32)
        if len(x) < min_samples:
            continue
        sf.write(outdir / f"{prefix}_{n:05d}.wav", x, 16000, subtype="PCM_16")
        n += 1
    return n

# min_samples=1: impulse responses are naturally short - most of the ~271 MIT
# RIRs are under a second, so the 1 s floor (right for background NOISE, which
# needs real duration) silently discarded all but 46 of them.
if len(list(RIR_DIR.glob("*.wav"))) < 100:
    rirs = load_dataset("davidscripka/MIT_environmental_impulse_responses",
                        split="train", streaming=True)
    print(f"RIRs written: {dump_16k(rirs, RIR_DIR, 1000, 'rir', min_samples=1)}")

if len(list(BG_DIR.glob('*.wav'))) < N_BACKGROUND_CLIPS * 0.9:
    # AudioSet spans speech, ambient and music, and its quota alone fills the
    # target - the Step 6 gate must never depend on the music source. The extra
    # FMA-derived music on top keeps dedicated music coverage (galleries play
    # music). mo-mittal/fma_small_genres vanished from the Hub; lewtun/music_genres
    # is the same FMA audio, parquet-backed, so it streams without a loader script.
    got = 0
    for repo, split, prefix, quota in [
        ("agkphysics/AudioSet", "train", "audioset", N_BACKGROUND_CLIPS),
        ("lewtun/music_genres", "train", "fma", N_BACKGROUND_CLIPS // 2),
    ]:
        try:
            ds = load_dataset(repo, split=split, streaming=True)
            got += dump_16k(ds, BG_DIR, quota, prefix)
            print(f"  {repo}: running total {got}")
        except Exception as e:
            print(f"  SKIPPED {repo}: {type(e).__name__}: {e}")
    print(f"background clips written: {got}")

print(f"\nRIRs:        {len(list(RIR_DIR.glob('*.wav')))}")
print(f"backgrounds: {len(list(BG_DIR.glob('*.wav')))}")

### Optional — add your own gallery ambient

Once Phase 2 is done (`record_ambient.py` at the exhibit site), drop those clips in
as background augmentation. Training against the actual room is the single highest-value
thing you can do for real-world false-accept rate. Skip this cell until you have them.

In [ ]:
# Upload a zip of ambient wavs recorded with record_ambient.py, then run this.
USE_MY_AMBIENT = False   # flip to True once you have gallery recordings

if USE_MY_AMBIENT:
    import zipfile, shutil, pathlib
    from google.colab import files
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("/content/my_ambient")
    n = 0
    for w in pathlib.Path("/content/my_ambient").rglob("*.wav"):
        shutil.copy(w, BG_DIR / f"gallery_{n:05d}.wav"); n += 1
    print(f"added {n} gallery ambient clips to the background pool")
else:
    print("skipped - no gallery ambient yet (Phase 2)")

---
## Step 6 — Verify before training

The repo learned this the hard way: a truncated download does not announce itself,
it just produces a silently worse model. All four must pass.

In [ ]:
import pathlib

EXPECTED_NEG_FEATURE_BYTES = 17_280_000_128   # exact size of the 2000 hr file

checks = []
p = pathlib.Path(NEGATIVE_FEATURES)
checks.append(("negative features exact size", p.exists() and p.stat().st_size == EXPECTED_NEG_FEATURE_BYTES,
               f"{p.stat().st_size:,} bytes (want {EXPECTED_NEG_FEATURE_BYTES:,})" if p.exists() else "missing"))
p = pathlib.Path(VALIDATION_FEATURES)
checks.append(("validation features present", p.exists() and p.stat().st_size > 1e8,
               f"{p.stat().st_size/1e9:.2f} GB" if p.exists() else "missing"))
n_rir = len(list(RIR_DIR.glob("*.wav")))
checks.append(("RIRs >= 100", n_rir >= 100, f"{n_rir} files"))
n_bg = len(list(BG_DIR.glob("*.wav")))
checks.append(("backgrounds >= 200", n_bg >= 200, f"{n_bg} files"))

print(f"{'check':<32}{'':<6}detail")
print("-" * 70)
for name, ok, detail in checks:
    print(f"{name:<32}{'PASS' if ok else 'FAIL':<6}{detail}")

failed = [n for n, ok, _ in checks if not ok]
if failed:
    raise SystemExit(f"\nSTOP. Fix these before training: {', '.join(failed)}\n"
                     "A short negative-features file trains a quietly worse model "
                     "rather than erroring.")
print("\nall checks pass - safe to train")

---
## Step 7 — Write the training config

In [ ]:
import yaml, pathlib

config = {
    "target_phrase":  TARGET_PHRASES,
    "custom_negative_phrases": CUSTOM_NEGATIVE_PHRASES,
    "model_name":     MODEL_NAME,
    "model_type":     MODEL_TYPE,
    "layer_size":     LAYER_SIZE,

    "n_samples":      N_SAMPLES,
    "n_samples_val":  N_SAMPLES_VAL,
    "tts_batch_size": 50,
    "piper_sample_generator_path": "/content/piper-sample-generator",

    "output_dir":     "/content/oww_output",
    "rir_paths":      [str(RIR_DIR)],
    "background_paths": [str(BG_DIR)],
    "background_paths_duplication_rate": [1],

    "augmentation_rounds":    AUG_ROUNDS,
    "augmentation_batch_size": 16,

    "feature_data_files": {"ACAV100M_sample": NEGATIVE_FEATURES},
    "false_positive_validation_data_path": VALIDATION_FEATURES,

    "batch_n_per_class": {
        "ACAV100M_sample":     1024,
        "adversarial_negative":  50,
        "positive":              50,
    },

    "steps":              STEPS,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": TARGET_FP_PER_HOUR,

    "total_length": 32000,   # overwritten by train.py from the generated clips
}

pathlib.Path("/content/oww_output").mkdir(exist_ok=True)
CONFIG_PATH = "/content/hey_cee_jap.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(open(CONFIG_PATH).read())

### Known traps — patch the trainer before running it

**DataLoader workers.** openWakeWord's trainer builds its DataLoader with
`num_workers=n_cpus, prefetch_factor=16`. On a small `/dev/shm` those workers die
with bus errors, or — worse — the run hangs at `Training: 0%` with no error at
all. That hang cost this project two days on WSL2. Colab usually has enough
shared memory, but `num_workers=0` costs almost nothing here (the data is
memory-mapped, not decoded) and removes the failure mode entirely.

**tflite feature embedding.** Every `AudioFeatures` call in the trainer defaults
to `inference_framework="tflite"`, and `tflite-runtime` cannot be installed on
Python ≥ 3.12 (see Step 1). The cell below redirects those calls to the
equivalent `.onnx` melspectrogram/embedding models. It first downloads the
feature models, which the pip package does not bundle.

**Pickled checkpoints vs torch ≥ 2.6.** `torch.load` now defaults to
`weights_only=True`, which rejects checkpoints containing pickled class
instances. Two checkpoints in this pipeline do: piper's voice model (patched in
Step 2) and deep-phonemizer's `en_us_cmudict_forward.pt`, which pickles its
`Preprocessor` (patched below — its last release is from 2023 and predates the
torch change). Both are pinned artifacts from their upstream releases, so
loading them fully is the intended behaviour.

**torchaudio ≥ 2.9 removed `torchaudio.info`.** torch-audiomentations (last
release Jan 2025) still calls it when sizing background-noise clips, which
crashes mid-augmentation with `AttributeError: module 'torchaudio' has no
attribute 'info'`. The patch reroutes that one metadata read through soundfile.
`torchaudio.load` still exists, so audio loading itself is untouched, and
openwakeword's own `torchaudio.info` helpers are not on the training path.

In [ ]:
import openwakeword, openwakeword.utils, pathlib

pkg = pathlib.Path(openwakeword.__file__).parent

try:
    import dp.model.model
    import torch_audiomentations.utils.io
except ImportError as e:
    raise SystemExit(f"missing package ({e}) - re-run the Step 1 install cell, "
                     "then run this cell again")
dp_model_py = pathlib.Path(dp.model.model.__file__)
tam_io_py = pathlib.Path(torch_audiomentations.utils.io.__file__)

# Melspectrogram + embedding models; every AudioFeatures below expects them in
# resources/models, and the pip package ships without them. Passing one small
# model name keeps this from downloading every official model.
openwakeword.utils.download_models(model_names=["hey_jarvis_v0.1"])

patches = [
    # DataLoader workers die/hang on small /dev/shm (see above).
    (pkg / "train.py", "num_workers=n_cpus, prefetch_factor=16", "num_workers=0"),
    # tflite default -> onnx: no tflite runtime exists for this Python (Step 1).
    (pkg / "train.py", "AudioFeatures(device='cpu', ncpu=4)",
                       "AudioFeatures(device='cpu', ncpu=4, inference_framework='onnx')"),
    (pkg / "train.py", "openwakeword.utils.AudioFeatures(device='cpu')",
                       "openwakeword.utils.AudioFeatures(device='cpu', inference_framework='onnx')"),
    (pkg / "utils.py", "F = AudioFeatures(device=device)",
                       "F = AudioFeatures(device=device, inference_framework='onnx')"),
    # deep-phonemizer's checkpoint pickles a Preprocessor object - same
    # torch>=2.6 weights_only wall as the piper voice in Step 2, same fix.
    (dp_model_py, "torch.load(checkpoint_path, map_location=device)",
                  "torch.load(checkpoint_path, map_location=device, weights_only=False)"),
    # torchaudio >= 2.9 removed torchaudio.info; torch-audiomentations still
    # uses it to size background clips. Read the header via soundfile instead
    # (the original function body after this line becomes dead code).
    (tam_io_py, "info = torchaudio.info(str(file_path))",
                "import soundfile as _sf; _i = _sf.info(str(file_path)); return (_i.frames, _i.samplerate)"),
    # Something in the trainer's import tree sets the root logger to DEBUG, and
    # onnxscript then floods the ONNX export with tens of thousands of DEBUG
    # lines, bloating the notebook. Clamp that one logger; INFO progress stays.
    (pkg / "train.py", "import logging",
                       "import logging; logging.getLogger('onnxscript').setLevel(logging.WARNING)"),
    # torch's new exporter splits weights into a <name>.onnx.data sidecar by
    # default; a downloaded .onnx alone then fails with 'External data path
    # does not exist'. Force a single self-contained file.
    (pkg / "train.py", 'os.path.join(output_dir, model_name + ".onnx"), opset_version=13)',
                       'os.path.join(output_dir, model_name + ".onnx"), opset_version=13, external_data=False)'),
]
for path, old, new in patches:
    src = path.read_text()
    if new in src:
        print(f"already patched: {path.name}: {new[:48]}...")
    elif old in src:
        path.write_text(src.replace(old, new))
        print(f"patched {path.name}: {old[:48]}...")
    else:
        print(f"WARNING: pattern not found in {path.name}: {old[:48]}... "
              f"(upstream changed? training may fail)")

---
## Step 8 — Train

Three passes, run separately so a failure tells you *which* stage broke:

1. **generate** — Piper synthesises the positive and adversarial-negative utterances
2. **augment** — applies RIRs + background noise, computes embeddings
3. **train** — fits the classifier head and exports ONNX

> **Expect the last cell to end with an error about `onnx_tf` / `tensorflow`.**
> That is `convert_onnx_to_tflite()` running *after* the ONNX file is already
> written. The `.onnx` is what this project needs; the tflite conversion is not
> worth installing a conflicting TF stack for. Step 9 confirms the model on disk.

> **Expected noise in the augment/train logs:** red onnxruntime errors about
> `CUDAExecutionProvider` / `libcublasLt.so.13` mean the current
> `onnxruntime-gpu` build targets CUDA 13 while Colab's stack is CUDA 12 — ORT
> falls back to CPU for feature embedding, which is correct, just slower. The
> torch-audiomentations `FutureWarning`s about `output_type` are harmless too.

In [ ]:
!cd /content && python -m openwakeword.train --training_config {CONFIG_PATH} --generate_clips

In [ ]:
!cd /content && python -m openwakeword.train --training_config {CONFIG_PATH} --augment_clips --overwrite

In [ ]:
!cd /content && python -m openwakeword.train --training_config {CONFIG_PATH} --train_model

In [ ]:
import pathlib
onnx_path = pathlib.Path(f"/content/oww_output/{MODEL_NAME}.onnx")
assert onnx_path.exists(), (
    "No .onnx was produced. The tflite error at the end of the previous cell is "
    "harmless, but an earlier failure is not - scroll up for the real traceback."
)
print(f"{onnx_path}  ({onnx_path.stat().st_size/1024:.0f} KB)")

---
## Step 9 — Score against the real recordings

This is the part that matters. Everything above optimised against synthetic audio;
this measures the model on 34 human utterances it has never seen, and on the 8
spelled-letter clips that **WW-5 says must stay silent**.

Recall is reported per SNR tier because the set is not uniform — an aggregate number
is dominated by the easy clips and can sit above 90% while every hard clip fails.

In [ ]:
import json, wave, pathlib
import numpy as np
from openwakeword.model import Model

FRAME = 1280
TIERS = ["easy", "moderate", "hard"]

manifest = json.load(open("/content/data/validation_manifest.json"))
model = Model(wakeword_models=[str(onnx_path)], inference_framework="onnx")

def load_int16(path):
    with wave.open(str(path)) as w:
        assert w.getframerate() == 16000, f"{path}: {w.getframerate()} Hz"
        a = np.frombuffer(w.readframes(w.getnframes()), dtype="<i2")
        if w.getnchannels() > 1:
            a = a.reshape(-1, w.getnchannels()).mean(axis=1).astype(np.int16)
    return a

def peak_score(audio):
    # Pad with 2 s leading / 0.5 s trailing silence. The model scores a stream
    # through a 2 s feature window, so a file shorter than the window ends
    # before the phrase is seen in full context and scores ~0 regardless of
    # content (measured: verified 1 s clips at 0.002 raw vs 0.742 padded).
    # The kiosk hears a continuous stream, so the padded number is the
    # deployment-realistic one. Same fix lives in CJAP/validate.py.
    audio = np.concatenate([np.zeros(2 * 16000, dtype=audio.dtype), audio,
                            np.zeros(16000 // 2, dtype=audio.dtype)])
    if hasattr(model, "reset"):
        model.reset()
    best = 0.0
    for i in range(0, len(audio) - FRAME + 1, FRAME):
        best = max(best, max(model.predict(audio[i:i + FRAME]).values()))
    return float(best)

for c in manifest["clips"]:
    c["score"] = peak_score(load_int16(pathlib.Path("/content", c["path"])))

pos = [c for c in manifest["clips"] if c["label"] == "positive"]
neg = [c for c in manifest["clips"] if c["label"] == "negative"]
json.dump(manifest, open("/content/scored_manifest.json", "w"), indent=2)
print(f"scored {len(pos)} positives and {len(neg)} negatives")

In [ ]:
# Pick the threshold: highest recall that still fires on none of the WW-5 negatives.
grid = np.round(np.arange(0.05, 1.0, 0.05), 2)
rows = []
for t in grid:
    rows.append((t,
                 sum(c["score"] >= t for c in pos) / len(pos),
                 sum(c["score"] >= t for c in neg),
                 {tier: (lambda g: sum(c["score"] >= t for c in g) / len(g) if g else float("nan"))
                        ([c for c in pos if c["tier"] == tier]) for tier in TIERS}))

print(f"{'thresh':>7}{'recall':>9}{'easy':>8}{'mod':>8}{'hard':>8}   WW-5 negatives firing")
print("-" * 66)
for t, rec, nfire, per in rows:
    flag = "  <-- clean" if nfire == 0 else ""
    print(f"{t:>7.2f}{rec:>8.0%}{per['easy']:>8.0%}{per['moderate']:>8.0%}"
          f"{per['hard']:>8.0%}{nfire:>8}{flag}")

clean = [r for r in rows if r[2] == 0]
if clean:
    best = max(clean, key=lambda r: r[1])
    print(f"\nRECOMMENDED THRESHOLD: {best[0]:.2f}  ->  recall {best[1]:.0%}, "
          f"zero false accepts on the spelled-letter negatives")
else:
    print("\nNo threshold rejects every spelled-letter negative. The model has not "
          "learned the WW-5 boundary - add more spelled-letter variants to "
          "CUSTOM_NEGATIVE_PHRASES and retrain.")

In [ ]:
THRESHOLD = 0.5   # set from the table above

print(f"threshold {THRESHOLD}\n")
print(f"{'clip':<22}{'speaker':<10}{'tier':<10}{'SNR':>6}{'score':>8}   result")
print("-" * 74)
for c in sorted(manifest["clips"], key=lambda c: (c["label"] != "positive", c["snr_db"])):
    fired = c["score"] >= THRESHOLD
    if c["label"] == "positive":
        mark = "hit " if fired else "MISS"
    else:
        mark = "FALSE ACCEPT (WW-5 breach)" if fired else "ok"
    print(f"{pathlib.Path(c['path']).name:<22}{c['speaker']:<10}{c['tier']:<10}"
          f"{c['snr_db']:>6.1f}{c['score']:>8.3f}   {mark}")

print("\n--- recall by tier ---")
for tier in TIERS:
    g = [c for c in pos if c["tier"] == tier]
    if g:
        hit = sum(c["score"] >= THRESHOLD for c in g)
        print(f"  {tier:<10}{hit/len(g):>6.0%}  ({hit}/{len(g)})   "
              f"{min(c['snr_db'] for c in g):.1f}-{max(c['snr_db'] for c in g):.1f} dB")

print("\n--- recall by speaker ---")
for s in sorted({c["speaker"] for c in pos}):
    g = [c for c in pos if c["speaker"] == s]
    hit = sum(c["score"] >= THRESHOLD for c in g)
    print(f"  {s:<10}{hit/len(g):>6.0%}  ({hit}/{len(g)})")

nfire = sum(c["score"] >= THRESHOLD for c in neg)
print(f"\n--- WW-5 spelled-letter negatives ---")
print(f"  {nfire}/{len(neg)} fired" + ("   PASS" if nfire == 0 else "   FAIL - 'see jay' must stay silent"))
print(f"\n  Resolution floor is {100/len(neg):.0f}% with {len(neg)} negatives. This says "
      f"nothing about\n  false accepts per hour in a real room - that needs Phase 2 ambient audio.")

---
## Step 10 — Download the model

In [ ]:
import shutil, json
from google.colab import files

bundle = f"/content/{MODEL_NAME}_bundle"
pathlib.Path(bundle).mkdir(exist_ok=True)
shutil.copy(onnx_path, bundle)

# torch's new exporter can split weights into a sidecar; the .onnx is useless
# without it. The patch cell forces a single file, but belt and braces here.
data_sidecar = pathlib.Path(str(onnx_path) + ".data")
if data_sidecar.exists():
    shutil.copy(data_sidecar, bundle)
    print(f"NOTE: bundling weight sidecar {data_sidecar.name} - keep it next to "
          f"the .onnx wherever it goes")

shutil.copy(CONFIG_PATH, bundle)
shutil.copy("/content/scored_manifest.json", bundle)

json.dump({
    "model_name": MODEL_NAME,
    "target_phrases": TARGET_PHRASES,
    "custom_negative_phrases": CUSTOM_NEGATIVE_PHRASES,
    "quick_smoke": QUICK_SMOKE,
    "n_samples": N_SAMPLES, "steps": STEPS, "layer_size": LAYER_SIZE,
    "target_fp_per_hour": TARGET_FP_PER_HOUR,
    "recommended_threshold": THRESHOLD,
    "validation": {
        "positives": len(pos), "negatives": len(neg),
        "recall": sum(c["score"] >= THRESHOLD for c in pos) / len(pos),
        "ww5_negatives_fired": int(sum(c["score"] >= THRESHOLD for c in neg)),
    },
}, open(f"{bundle}/training_record.json", "w"), indent=2)

shutil.make_archive(bundle, "zip", bundle)
files.download(f"{bundle}.zip")
print(f"downloading {MODEL_NAME}_bundle.zip")

---
## Step 11 — Wire it into the app

Unzip into `wakeword/CJAP/models/`, then verify locally against a live mic:

```bash
cd "wakeword/CJAP"
./.venv/Scripts/python.exe validate.py --model models/hey_cee_jap.onnx --threshold 0.5
./.venv/Scripts/python.exe test_detection.py --model models/hey_cee_jap.onnx --device <mic> --threshold 0.5
```

Then point the app at it:

```bash
export CJ_WAKE_OWW_MODEL_PATH="wakeword/CJAP/models/hey_cee_jap.onnx"
export CJ_WAKE_BACKEND=openwakeword
```

### Two things this notebook does *not* finish

1. **`OpenWakeWordDetector.detect()` is still a stub.** It raises `NotImplementedError`
   at [`app/wake_word.py`](../../../app/wake_word.py) — setting `CJ_WAKE_BACKEND=openwakeword`
   will crash until someone implements it against the trained model. A model is
   necessary but not sufficient.
2. **False-accepts-per-hour is still unmeasured.** 8 negative clips cannot resolve
   below 12%; a usable target is far lower. That number only comes from Phase 2 —
   an hour of real gallery ambient, replayed through `test_detection.py --duration 3600 --log`.
   Note that flag lives in the *outer* `wakeword/test_detection.py`, not the copy in `CJAP/`.